# Chess Policy Model — Data Preparation

This notebook prepares a clean supervised-learning dataset for the ChessMind Arena chess policy model.

The source dataset is:

`Lichess/chess-position-evaluations`

Each source record contains a chess position, Stockfish principal variation, search depth, node count, and evaluation.

Our target representation is:

    Chess position → strong candidate move

The prepared dataset will contain:

- FEN
- target move
- Stockfish evaluation
- mate evaluation
- search depth
- node count

The first dataset will contain 100,000 positions.

The dataset will be split into:

- 80% training
- 10% validation
- 10% test

The prepared data will be saved as Parquet files under:

    ml/data/

The raw Lichess dataset will NOT be downloaded completely.

In [2]:
!pip install -q datasets python-chess pandas pyarrow tqdm

In [3]:
import chess
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

In [10]:
SEED = 42

TARGET_SIZE = 100000

TRAIN_SIZE = int(TARGET_SIZE * 0.8)
VALID_SIZE = int(TARGET_SIZE * 0.1)
TEST_SIZE = int(TARGET_SIZE * 0.1)

MIN_DEPTH = 30

In [11]:
dataset = load_dataset(
    "Lichess/chess-position-evaluations",
    split=f"train",
    streaming=True,
)

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

In [12]:
dataset = dataset.shuffle(seed=SEED, buffer_size=TARGET_SIZE)

In [7]:
def extract_training_example(row):
    try:
        fen = row["fen"]
        line = row["line"]

        if not fen or not line:
            return None

        board = chess.Board(fen)

        pv_moves = line.split()

        if not pv_moves:
            return None

        target_move = pv_moves[0]

        move = chess.Move.from_uci(target_move)

        if move not in board.legal_moves:
            return None

        return {
            "fen": fen,
            "target_move": target_move,
            "cp": row["cp"],
            "mate": row["mate"],
            "depth": row["depth"],
            "knodes": row["knodes"],
        }

    except Exception:
        return None

In [13]:
examples = []

MAX_SCANNED = 2000000
count = 0
seen_fens = set()

for row in tqdm(dataset, total=MAX_SCANNED):
    if count >= MAX_SCANNED:
        break
    
    if len(examples) >= TARGET_SIZE:
        break

    count += 1
    depth = row["depth"]

    if depth is None or depth < MIN_DEPTH:
        continue

    fen = row["fen"]

    if not fen or fen in seen_fens:
        continue

    example = extract_training_example(row)
    
    example = extract_training_example(row)
    
    if not example:
        continue
    
    seen_fens.add(fen)
    examples.append(example)

  0%|          | 0/2000000 [00:00<?, ?it/s]

In [14]:
print("Examples collected:", len(examples))
print("Unique FENs:", len(seen_fens))

Examples collected: 100000
Unique FENs: 100000


In [15]:
df = pd.DataFrame(examples)

print("DataFrame shape:", df.shape)
df.head()

DataFrame shape: (100000, 6)


,fen,target_move,cp,mate,depth,knodes
0,8/8/8/8/8/4kpK1/3r4/R7 b - -,f3f2,0.0,NaN,60,18057
1,8/6QR/p2p1Pp1/1n1P3p/1P2k3/6P1/P4PKP/8 w - -,f6f7,NaN,6.0,65,6328
2,8/8/8/6p1/5k1p/1R3P2/6K1/r7 b - -,a1d1,-5.0,NaN,32,3621
3,4Q3/p1p4q/1p1p4/3P3k/2P3p1/8/P4P2/2B3K1 b - -,h7g6,NaN,2.0,85,996
4,r2q1bk1/pp3p2/2ppr2p/4n1p1/2P1P3/P1N1B2P/1P2QP...,b2b3,10.0,NaN,30,280287


In [16]:
df.isna().sum()

,0
fen,0
target_move,0
cp,24801
mate,75199
depth,0
knodes,0


In [17]:
df["depth"].describe()

,depth
count,100000.000000
mean,41.485570
std,17.690769
min,30.000000
25%,32.000000
50%,36.000000
75%,46.000000
max,245.000000


In [18]:
df["depth"].value_counts().sort_index()

,count
depth,
30,20895
32,13929
34,9956
36,7772
38,6190
40,5572
42,4841
44,4544
46,4039


In [19]:
df["cp"].describe()

,cp
count,75199.000000
mean,53.772098
std,1468.720630
min,-20000.000000
25%,0.000000
50%,0.000000
75%,14.000000
max,20000.000000


In [21]:
print("Mate positions:", df["mate"].notna().sum())
print("Normal evaluations:", df["cp"].notna().sum())

Mate positions: 24801
Normal evaluations: 75199


In [22]:
df[df["mate"].notna()].head()

,fen,target_move,cp,mate,depth,knodes
1,8/6QR/p2p1Pp1/1n1P3p/1P2k3/6P1/P4PKP/8 w - -,f6f7,NaN,6.0,65,6328
3,4Q3/p1p4q/1p1p4/3P3k/2P3p1/8/P4P2/2B3K1 b - -,h7g6,NaN,2.0,85,996
9,3k4/5P1R/8/p1pPp1p1/Pb2P3/3b2K1/8/8 b - -,d3b5,NaN,3.0,32,3727
11,5k2/4rp2/8/2Q5/5K2/8/8/8 b - -,f7f5,NaN,22.0,36,81919
13,7k/2K5/6P1/7p/p7/1P6/8/8 w - -,c7c6,NaN,-11.0,38,44739


In [23]:
def validate_target(row):
    try:
        board = chess.Board(row["fen"])
        move = chess.Move.from_uci(row["target_move"])

        return move in board.legal_moves

    except Exception:
        return False

In [24]:
df["target_is_legal"] = df.apply(validate_target, axis=1)

print(df["target_is_legal"].value_counts())

target_is_legal
True    100000
Name: count, dtype: int64


In [25]:
assert df["target_is_legal"].all()

In [26]:
df = df.drop(columns=["target_is_legal"])

In [27]:
for _, row in df.head(10).iterrows():
    print("FEN:", row["fen"])
    print("Move:", row["target_move"])
    print("CP:", row["cp"])
    print("Mate:", row["mate"])
    print("Depth:", row["depth"])
    print("-" * 80)

FEN: 8/8/8/8/8/4kpK1/3r4/R7 b - -
Move: f3f2
CP: 0.0
Mate: nan
Depth: 60
--------------------------------------------------------------------------------
FEN: 8/6QR/p2p1Pp1/1n1P3p/1P2k3/6P1/P4PKP/8 w - -
Move: f6f7
CP: nan
Mate: 6.0
Depth: 65
--------------------------------------------------------------------------------
FEN: 8/8/8/6p1/5k1p/1R3P2/6K1/r7 b - -
Move: a1d1
CP: -5.0
Mate: nan
Depth: 32
--------------------------------------------------------------------------------
FEN: 4Q3/p1p4q/1p1p4/3P3k/2P3p1/8/P4P2/2B3K1 b - -
Move: h7g6
CP: nan
Mate: 2.0
Depth: 85
--------------------------------------------------------------------------------
FEN: r2q1bk1/pp3p2/2ppr2p/4n1p1/2P1P3/P1N1B2P/1P2QPP1/R4RK1 w - -
Move: b2b3
CP: 10.0
Mate: nan
Depth: 30
--------------------------------------------------------------------------------
FEN: 8/8/5p2/8/3R1P2/3p1k1P/4rP2/3K4 b - -
Move: e2f2
CP: 0.0
Mate: nan
Depth: 40
----------------------------------------------------------------------------

In [28]:
move_counts = df["target_move"].value_counts()

move_counts.head(20)

,count
target_move,
g7g6,473
h2h3,470
h7h6,468
f7f6,444
g2g3,438
f2f3,434
h6h5,386
g6g5,385
h2h4,347


In [29]:
print("Unique target moves:", df["target_move"].nunique())

Unique target moves: 1890


In [30]:
def get_ply_from_fen(fen):
    board = chess.Board(fen)
    return board.fullmove_number, board.turn

In [31]:
def count_pieces(fen):
    board = chess.Board(fen)

    return len(board.piece_map())


df["piece_count"] = df["fen"].apply(count_pieces)

df["piece_count"].describe()

,piece_count
count,100000.000000
mean,15.030350
std,7.476413
min,3.000000
25%,9.000000
50%,14.000000
75%,21.000000
max,62.000000


In [32]:
df = df.drop(columns=["piece_count"])

In [33]:
df = df.sample(
    frac=1,
    random_state=SEED,
).reset_index(drop=True)

In [34]:
train_df = df.iloc[:TRAIN_SIZE].copy()
val_df = df.iloc[
    TRAIN_SIZE:TRAIN_SIZE + VALID_SIZE
].copy()

test_df = df.iloc[
    TRAIN_SIZE + VALID_SIZE:
].copy()

In [35]:
train_fens = set(train_df["fen"])
val_fens = set(val_df["fen"])
test_fens = set(test_df["fen"])

print("Train/Val overlap:", len(train_fens & val_fens))
print("Train/Test overlap:", len(train_fens & test_fens))
print("Val/Test overlap:", len(val_fens & test_fens))

Train/Val overlap: 0
Train/Test overlap: 0
Val/Test overlap: 0


In [49]:
from pathlib import Path

DATA_DIR = Path("/../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [48]:
train_df.to_parquet(
    DATA_DIR / "train.parquet",
    index=False,
)

val_df.to_parquet(
    DATA_DIR / "val.parquet",
    index=False,
)

test_df.to_parquet(
    DATA_DIR / "test.parquet",
    index=False,
)

In [42]:
train_df

,fen,target_move,cp,mate,depth,knodes
0,8/8/4K1k1/4PRp1/8/4rP2/8/8 w - -,f5f6,0.0,NaN,65,136722
1,8/8/1p6/2pk4/1P1P1p2/2K2P2/8/8 w - -,d4c5,0.0,NaN,70,46343
2,4k3/5p2/8/4P2p/2P4Q/4R3/q4P1K/8 w - -,h4f6,NaN,11.0,44,18902
3,8/2r3k1/p6p/1p6/2p5/P2R1KP1/8/8 b - -,c4d3,NaN,-10.0,44,144
4,8/R1r4p/2k1p1pb/4P3/4pP2/4B1P1/5K1P/8 b - -,c7a7,0.0,NaN,50,15119
...,...,...,...,...,...,...
79995,r1b1k2r/pp5p/6pn/1Np5/3nPp1q/7P/PP1B1P2/R2QKBR...,h6f5,-25.0,NaN,32,13841
79996,2kr4/ppp5/5n2/7p/2P3bP/5N2/P4PB1/4R1K1 b - -,c7c6,-7.0,NaN,40,59957
79997,4k3/pQ6/1p1NP1p1/5b2/7P/6P1/P2r1qBK/8 b - -,e8d8,NaN,1.0,65,27337
79998,6Q1/8/1K2k1p1/6P1/8/8/8/8 b - -,e6f5,NaN,10.0,36,2201


In [47]:
print(DATA_DIR/"val.parquet")

/../data/val.parquet


In [50]:
from pathlib import Path
import os

print("Current working directory:")
print(Path.cwd())

print("\nFiles here:")
print(os.listdir("."))

Current working directory:
/content

Files here:
['.config', 'sample_data']


In [51]:
import os

for file in [
    DATA_DIR / "train.parquet",
    DATA_DIR / "val.parquet",
    DATA_DIR / "test.parquet",
]:
    print(
        file,
        f"{os.path.getsize(file) / (1024 ** 2):.2f} MB"
    )

/../data/train.parquet 2.96 MB
/../data/val.parquet 0.40 MB
/../data/test.parquet 0.41 MB


In [52]:
print("ChessMind Arena Policy Dataset")
print("=" * 40)

print(f"Train:      {len(train_df):,}")
print(f"Validation: {len(val_df):,}")
print(f"Test:       {len(test_df):,}")

print("\nColumns:")
print(train_df.columns.tolist())

print("\nMinimum depth:")
print(train_df["depth"].min())

print("\nMaximum depth:")
print(train_df["depth"].max())

print("\nUnique target moves:")
print(train_df["target_move"].nunique())

ChessMind Arena Policy Dataset
Train:      80,000
Validation: 10,000
Test:       10,000

Columns:
['fen', 'target_move', 'cp', 'mate', 'depth', 'knodes']

Minimum depth:
30

Maximum depth:
245

Unique target moves:
1880
